# INF-473 - Introducción a la Inteligencia Artificial Explicable
## 1er Semestre 2026 -- Prof. Raquel Pezoa
<center>
    <h1> Contrafactuales </h2>
</center>

* Se usan redes neuronales de Pytorch
* El dataset MNIST
* Contrafactuales de [OmniXAI](https://opensource.salesforce.com/OmniXAI/latest/index.html)
* Se usa la función CounterfactualExplainer, la cual:
  * Soporta solo tareas de clasificación.
  * Corresponde al método de Wachter, presentado en el paper "Counterfactual Explanations without Opening the Black Box: Automated Decisions and the GDPR" Sandra Wachter, Brent Mittelstadt, Chris Russell. [Link](https://arxiv.org/pdf/1711.00399)

## Bibliotecas

In [11]:
import torch
import torch.nn as nn
import torchvision # datasets (como MNIST)
                # utilidades para imágenes
import torchvision.transforms as transforms #convertir imágenes a tensores
                                            # normalizar
                                            # preprocesar datos
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from omnixai.data.image import Image #representar imágenes en su formato interno
                                    # conectar con el CounterfactualExplainer

## Modelo Convolucional

In [12]:
class InputData(Dataset): # dataset compatbile con Pytorch

    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return self.images.shape[0]

    def __getitem__(self, index):
        return self.images[index], self.labels[index]


class MNISTNet(nn.Module):

    def __init__(self):
        super().__init__()
        # capas convolucionales, entrada 1 canal, 10 feature maps, kernel 5x5
        self.conv_layers = nn.Sequential( 
            nn.Conv2d(1, 10, kernel_size=5),
            nn.MaxPool2d(2),
            nn.ReLU(),
            nn.Conv2d(10, 20, kernel_size=5),
            nn.Dropout(),
            nn.MaxPool2d(2),
            nn.ReLU(),
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(320, 50),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(50, 10) # salida de 10 digitos
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = torch.flatten(x, 1)
        x = self.fc_layers(x)
        return x

## Formateo de Datos

In [13]:
# Load the training and test datasets
train_data = torchvision.datasets.MNIST(root='../data', train=True, download=True)
test_data = torchvision.datasets.MNIST(root='../data', train=False, download=True)
train_data.data = train_data.data.numpy()
test_data.data = test_data.data.numpy()

class_names = (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)
# Use `Image` objects to represent the training and test datasets en formato OmniXAI
x_train, y_train = Image(train_data.data, batched=True), train_data.targets
x_test, y_test = Image(test_data.data, batched=True), test_data.targets


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Crea el modelo
model = MNISTNet().to(device)

# Transformación de imagenes
transform = transforms.Compose([transforms.ToTensor()])

# Para cada imagen im convierte desde formato OmniXAI a PIL (tensor Pytorch) y junta todo en un batch (stack)
preprocess = lambda ims: torch.stack([transform(im.to_pil()) for im in ims])

## Entrenamiento

In [15]:
learning_rate=1e-3
batch_size=128
num_epochs=10

train_loader = DataLoader(
    dataset=InputData(preprocess(x_train), y_train),
    batch_size=batch_size,
    shuffle=True
)
test_loader = DataLoader(
    dataset=InputData(preprocess(x_test), y_test),
    batch_size=batch_size,
    shuffle=False
)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_func = nn.CrossEntropyLoss()

model.train()
for epoch in range(num_epochs):
    for i, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        loss = loss_func(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

correct_pred = {name: 0 for name in class_names}
total_pred = {name: 0 for name in class_names}

model.eval()
for x, y in test_loader:
    images, labels = x.to(device), y.to(device)
    outputs = model(images)
    _, predictions = torch.max(outputs, 1)
    for label, prediction in zip(labels, predictions):
        if label == prediction:
            correct_pred[class_names[label]] += 1
        total_pred[class_names[label]] += 1

for name, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[name]
    print("Accuracy for class {} is: {:.1f} %".format(name, accuracy))

NameError: name 'transform' is not defined

## Explicación Contrafactual

Documentación oficial en [OmniXAI](https://opensource.salesforce.com/OmniXAI/latest/omnixai.explainers.vision.counterfactual.html#module-omnixai.explainers.vision.counterfactual.ce)


In [ ]:
from omnixai.explainers.vision import CounterfactualExplainer

explainer = CounterfactualExplainer(
    model=model,
    preprocess_function=preprocess
)

/Users/rpezoa/Projects/teaching/omnixai_mnist_cf/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
explanations = explainer.explain(x_test[0:5])

Binary step: 5 |███████████████████████████████████████-| 99.0% 

In [8]:
explanations.ipython_plot(index=4)
